# Explot Demo

**Explot** is a trust-aware automated first-pass analyst for tabular data — one CSV in, one HTML report out.

This notebook installs Explot, downloads the telco churn dataset, runs the full 9-stage pipeline, and displays the findings.

**Pipeline:** profiling → exploration → preprocessing → dimensionality (PCA) → unsupervised → supervised (SHAP) → findings

In [ ]:
!pip install -q "git+https://github.com/melisgncl/explot.git[ml,survival]"

In [ ]:
!wget -q https://raw.githubusercontent.com/melisgncl/explot/main/data/telco_churn.csv
import pandas as pd
df = pd.read_csv('telco_churn.csv')
print(f'Shape: {df.shape}  |  Churn rate: {df["Churn"].value_counts(normalize=True)["Yes"]:.1%}')
df.head(3)

In [ ]:
!explot telco_churn.csv -o report.html --fast --target Churn --task classification -q
print('Done — report.html generated')

In [ ]:
!explot telco_churn.csv -o results.json --json --fast --target Churn --task classification -q

import json
with open('results.json') as f:
    results = json.load(f)

findings = results.get('findings', {}).get('outputs', {}).get('findings', [])
verdict = results.get('findings', {}).get('outputs', {}).get('verdict', 'UNKNOWN')

print(f'\n=== VERDICT: {verdict} ===\n')
for finding in findings[:8]:
    conf = finding.get('confidence', '')
    msg = finding.get('message', '')
    print(f'[{conf}] {msg}')

In [ ]:
best_models = results.get('supervised', {}).get('outputs', {}).get('best_models', {})
for target, info in best_models.items():
    score = info.get('mean', 0)
    model = info.get('model', '?')
    metric = info.get('metric', '?')
    flags = info.get('trust_flags', [])
    shap = info.get('shap_importance', [])[:3]
    print(f"Target : '{target}'")
    print(f"  Best  : {model} ({metric} = {score:.3f})")
    print(f"  Flags : {flags if flags else 'none'}")
    if shap:
        print(f"  SHAP  : {[s['feature'] for s in shap]}")
    print()

In [ ]:
from IPython.display import IFrame
IFrame('report.html', width='100%', height=700)